# Functional Specialization Analysis

Load `specialization_metrics.json` (produced by `scripts/03_functional_specialization.py`) for one or more runs and plot metrics vs sparsity (or structural Q), Bená-style "Model Specialization" (y) vs "Sparsity (p)" (x).

## 1. Setup and paths

In [1]:
import os
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

project_root = Path(os.getcwd()).resolve()
while not (project_root / "a1b2").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Project root (containing a1b2) not found.")
    project_root = project_root.parent

data_folder = project_root / "data"
sim_folder = data_folder / "simulations"
print("Project root:", project_root)

Project root: /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular


## 2. Load specialization metrics for one or more runs

Specify `run_ids`: list of run folder names (e.g. `two_module_rnn_25`, `two_module_rnn_25_low_sparse`). For each run we need `specialization_metrics.json` and `settings.json` in the run folder.

In [2]:
# Change from 50 to 25 as standard size
run_ids = [ 
    "two_module_rnn_25",
    "two_module_rnn_25_low_sparse",
    "two_module_rnn_25_sp05",
]  # Customize: use run IDs that have specialization_metrics.json

rows = []
for run_id in run_ids:
    path = sim_folder / run_id
    metrics_file = path / "specialization_metrics.json"
    settings_file = path / "settings.json"
    if not metrics_file.exists():
        print(f"Skip {run_id}: no specialization_metrics.json")
        continue
    with open(metrics_file, "r") as f:
        metrics = json.load(f)
    sparsity = None
    if settings_file.exists():
        with open(settings_file, "r") as f:
            settings = json.load(f)
        cond = settings.get("condition", {})
        sparsity = cond.get("sparsity")
    mean = metrics.get("mean", {})
    rows.append({
        "run_id": run_id,
        "sparsity": sparsity,
        "retraining_specialization": mean.get("retraining_specialization", {}).get("mean"),
        "correlation_specialization": mean.get("correlation_specialization", {}).get("mean"),
        "ablation_specialization": mean.get("ablation_specialization", {}).get("mean"),
        "retraining_sem": mean.get("retraining_specialization", {}).get("sem"),
        "correlation_sem": mean.get("correlation_specialization", {}).get("sem"),
        "ablation_sem": mean.get("ablation_specialization", {}).get("sem"),
    })

df = __import__("pandas").DataFrame(rows)
df

Skip two_module_rnn_25: no specialization_metrics.json
Skip two_module_rnn_25_low_sparse: no specialization_metrics.json
Skip two_module_rnn_25_sp05: no specialization_metrics.json


""


## 3. Plot Model Specialization vs Sparsity

If sparsity is available, x-axis = sparsity; otherwise x-axis = run_id.

In [3]:
x_label = "sparsity" if df["sparsity"].notna().any() else "run_id"
x_vals = df["sparsity"].values if df["sparsity"].notna().any() else np.arange(len(df))
x_tick_labels = df["run_id"].tolist() if not df["sparsity"].notna().any() else None

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
metrics_names = ["retraining_specialization", "correlation_specialization", "ablation_specialization"]
sem_col_map = {"retraining_specialization": "retraining_sem", "correlation_specialization": "correlation_sem", "ablation_specialization": "ablation_sem"}
for ax, m in zip(axes, metrics_names):
    y = df[m].values
    sem_key = sem_col_map.get(m)
    err = df[sem_key].values if sem_key and sem_key in df.columns else None
    ax.errorbar(x_vals, y, yerr=err, fmt="o-", capsize=3)
    ax.set_ylabel("Model Specialization")
    ax.set_title(m.replace("_", " ").title())
    ax.set_xlabel(x_label)
    if x_tick_labels is not None:
        ax.set_xticks(x_vals)
        ax.set_xticklabels(x_tick_labels, rotation=45, ha="right")
    ax.set_ylim(0, None)
plt.tight_layout()
plt.show()

KeyError: 'sparsity'